# Extração dos vetores (protocolo de Lucy et al., 2022)

Para cada frase: o modelo lê a frase inteira; o vetor da palavra-alvo é a **média dos seus pedaços** (de todas as
ocorrências dela na frase), usando as **últimas 4 camadas concatenadas** (3.072 números). O vetor da palavra no
período é a **média dos vetores das frases**. Z-score e normalização L2 são feitos depois, no seu computador: **os vetores salvos aqui são brutos**
(nenhuma normalização aplicada), para que qualquer pós-processamento possa ser trocado sem rodar o modelo de novo.

O que fica salvo em cada `amostra_<período>.npz` / `completo_<ano>.npz`:
- `medias`: média dos vetores brutos das frases (o vetor principal);
- `n`: quantas frases entraram em cada palavra.

As 3.072 colunas são as camadas -4, -3, -2, -1 em sequência (768 cada): dá para usar só a última camada (colunas 2304 a 3071)
ou a média das 4 sem rodar de novo.

`zscore.npz` guarda os **vetores individuais** (uma palavra sorteada por frase, em float16), além da média e do desvio,
para que a estimativa do z-score possa ser recalculada ou testada (ex.: divisão em metades).

Há duas chaves na primeira célula de código:

**`ETAPA`** — qual rodada fazer:
- `"teste"`: só o BERT, só 2012 e 2018. Gera o método de Lucy (até 500 frases) **e** todas as frases (Jeito 1)
  das palavras de gênero, ocupações, inteligência e aparência, mais o z-score. Serve para comparar os dois jeitos.
- `"valendo"`: os 3 modelos, os 9 períodos, método de Lucy + z-score. Só depois que o teste aprovar.

**`CONFERENCIA`** — `True` roda poucas frases (alguns minutos) só para ver se o código funciona;
`False` roda tudo. Sempre faça uma conferência antes de cada rodada de verdade.

**Antes:** Accelerator = GPU, Internet = On, e o dataset com a pasta `contextos` adicionado em *Input*.

**Como rodar:** `CONFERENCIA = True` → Run All. Se der certo, `CONFERENCIA = False` →
**Save Version → Save & Run All (Commit)**: roda em segundo plano, pode fechar o navegador.

In [ ]:
ETAPA = "teste"            # "teste" (BERT, 2012 e 2018, os dois jeitos) ou "valendo" (3 modelos, 9 períodos)
CONFERENCIA = True         # True = poucas frases, só para ver se funciona; False = rodada de verdade
TODOS_MODELOS = ["bert-base-cased", "roberta-base", "answerdotai/ModernBERT-base"]
MODELO_VALIDACAO = "bert-base-cased"   # o único que roda o teste
ANOS_TESTE = ["2012", "2018"]
assert ETAPA in ("teste", "valendo"), 'ETAPA deve ser "teste" ou "valendo"'
MODELOS = [MODELO_VALIDACAO] if ETAPA == "teste" else TODOS_MODELOS
CAMADAS = 4                # últimas 4 camadas concatenadas (Devlin et al. 2019; Lucy et al. 2022)
LOTE = 128                 # frases por vez na GPU (diminui sozinho se faltar memória)
SEMENTE = 42

import glob, os, re, time, random, shutil
import numpy as np, pandas as pd, torch, transformers
from transformers import AutoModel, AutoTokenizer

assert torch.cuda.is_available(), "Sem GPU: ligue Accelerator = GPU (Session options) e rode de novo."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| transformers", transformers.__version__)

SAIDA = f"/kaggle/working/vetores_{ETAPA}" + ("_conferencia" if CONFERENCIA else "")
os.makedirs(SAIDA, exist_ok=True)

def achar(nome):
    r = glob.glob(f"/kaggle/input/**/{nome}", recursive=True)
    assert r, f"Não achei {nome}: adicione o dataset em 'Add Input'."
    return r[0]

amostra = pd.read_parquet(achar("contextos_amostra.parquet"))
frases_z = pd.read_parquet(achar("frases_zscore.parquet"))
completos = {ano: achar(f"contextos_completo_{ano}.parquet") for ano in ANOS_TESTE} if ETAPA == "teste" else {}
if ETAPA == "teste":
    amostra = amostra[amostra["periodo"].isin(ANOS_TESTE)]
if CONFERENCIA:
    amostra = amostra.groupby("periodo").head(200)
    frases_z = frases_z.head(300)
print(f"ETAPA = {ETAPA} | modelos: {MODELOS} | saída: {SAIDA}")
print(f"{len(amostra):,} frases na amostra | {len(frases_z):,} frases para o z-score")
print(amostra.groupby("periodo").size().to_string())

In [ ]:
LETRA_OU_DIGITO = r"[^\W_]"

def posicoes(frase, palavra):
    """Onde a palavra aparece na frase (todas as ocorrências), com a mesma regra de palavra do
    BasicTokenizer usado na amostragem: sem letra ou dígito colado antes ou depois."""
    padrao = rf"(?<!{LETRA_OU_DIGITO}){re.escape(palavra)}(?!{LETRA_OU_DIGITO})"
    return [(m.start(), m.end()) for m in re.finditer(padrao, frase, re.IGNORECASE)]

def carregar(nome):
    tok = AutoTokenizer.from_pretrained(nome)
    mod = AutoModel.from_pretrained(nome).to("cuda").eval().half()   # fp16: mesmo resultado, ~4x mais rápido
    return tok, mod

def processar(frases, alvos, tok, mod, acumular, rotulo=""):
    """frases: lista de textos. alvos[i]: lista de (chave, [(início, fim), ...]) da frase i.
    Para cada alvo, chama acumular(chave, vetor), onde vetor = média dos pedaços que se sobrepõem
    às ocorrências da palavra, nas últimas CAMADAS camadas concatenadas."""
    ordem = sorted(range(len(frases)), key=lambda i: len(frases[i]))   # tamanhos parecidos no mesmo lote
    lote, i, t0, perdidos, n_lotes = LOTE, 0, time.time(), 0, 0
    with torch.no_grad():
        while i < len(ordem):
            idx = ordem[i:i + lote]
            try:
                enc = tok([frases[k] for k in idx], return_offsets_mapping=True, truncation=True,
                          max_length=512, padding=True, return_tensors="pt")
                saida = mod(input_ids=enc["input_ids"].cuda(), attention_mask=enc["attention_mask"].cuda(),
                            output_hidden_states=True)
                vet = torch.cat(saida.hidden_states[-CAMADAS:], dim=-1)          # lote x tokens x 3072
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); lote = max(1, lote // 2)
                print(f"  memória cheia: lote reduzido para {lote}"); continue
            offs = enc["offset_mapping"].numpy()
            for b, k in enumerate(idx):
                ini, fim = offs[b, :, 0], offs[b, :, 1]
                valido = fim > ini
                for chave, spans in alvos[k]:
                    m = np.zeros(len(ini), dtype=bool)
                    for a0, a1 in spans:
                        m |= valido & (ini < a1) & (fim > a0)
                    if m.any():
                        tokens = torch.as_tensor(np.nonzero(m)[0], device="cuda")
                        acumular(chave, vet[b, tokens].float().mean(0).cpu().numpy().astype(np.float64))
                    else:
                        perdidos += 1      # palavra além do limite de 512 tokens
            i += len(idx); n_lotes += 1
            if n_lotes % 200 == 0 or i >= len(ordem):
                print(f"  {rotulo} {i:,}/{len(ordem):,} frases ({time.time() - t0:.0f} s)", flush=True)
    return perdidos

def vetores_por_palavra(df, tok, mod, rotulo):
    """Média dos vetores das frases, por palavra. df tem colunas frase e palavras."""
    frases = df["frase"].tolist()
    alvos = []
    for frase, palavras in zip(frases, df["palavras"]):
        alvos.append([(p, posicoes(frase, p)) for p in palavras.split()])
    alvos = [[(p, s) for p, s in a if s] for a in alvos]
    soma, n = {}, {}
    def acumular(chave, v):
        if chave in soma: soma[chave] += v; n[chave] += 1
        else: soma[chave] = v.copy(); n[chave] = 1
    perdidos = processar(frases, alvos, tok, mod, acumular, rotulo)
    palavras = sorted(soma)
    return dict(palavras=np.array(palavras),
                medias=np.array([soma[p] / n[p] for p in palavras], dtype=np.float32),         # BRUTO
                n=np.array([n[p] for p in palavras])), perdidos

def estatisticas_zscore(df, tok, mod):
    """Como Lucy et al.: uma palavra sorteada por frase; média e desvio padrão de cada dimensão.
    Também devolve os vetores individuais (float16), para recalcular ou testar a estimativa depois."""
    rng = random.Random(SEMENTE)
    frases, alvos, sorteadas, periodos = [], [], [], []
    tem_periodo = "periodo" in df.columns
    for i, frase in enumerate(df["frase"]):
        palavras = list(re.finditer(rf"{LETRA_OU_DIGITO}+", frase))
        if palavras:
            m = rng.choice(palavras)
            k = len(frases)
            frases.append(frase); alvos.append([(k, [(m.start(), m.end())])])
            sorteadas.append(m.group(0).lower())
            periodos.append(str(df["periodo"].iloc[i]) if tem_periodo else "")
    vetores = np.zeros((len(frases), 1), dtype=np.float16)
    ok = np.zeros(len(frases), dtype=bool)
    def acumular(k, v):
        nonlocal vetores
        if vetores.shape[1] == 1: vetores = np.zeros((len(frases), len(v)), dtype=np.float16)
        vetores[k] = v.astype(np.float16); ok[k] = True
    processar(frases, alvos, tok, mod, acumular, "z-score")
    v = vetores[ok].astype(np.float64)
    media = v.mean(0); desvio = np.maximum(v.std(0), 1e-6)
    return dict(media=media.astype(np.float32), desvio=desvio.astype(np.float32), n=int(ok.sum()),
                vetores=vetores[ok], palavra=np.array(sorteadas)[ok], periodo=np.array(periodos)[ok])

In [ ]:
registro = []
for nome in MODELOS:
    curto = nome.split("/")[-1]
    pasta = os.path.join(SAIDA, curto); os.makedirs(pasta, exist_ok=True)
    print(f"\n===== {nome} =====")
    tok, mod = carregar(nome)

    # 1. Jeito 2: frases sorteadas, um arquivo por período
    for periodo, df in amostra.groupby("periodo"):
        arq = os.path.join(pasta, f"amostra_{periodo}.npz")
        if os.path.exists(arq): continue
        t0 = time.time()
        res, perdidos = vetores_por_palavra(df, tok, mod, f"{curto} {periodo}")
        np.savez_compressed(arq, **res)
        registro.append({"modelo": curto, "etapa": f"amostra {periodo}", "frases": len(df),
                         "palavras": len(res["palavras"]), "perdidos": perdidos, "segundos": round(time.time() - t0)})

    # 2. estatísticas do z-score
    arq = os.path.join(pasta, "zscore.npz")
    if not os.path.exists(arq):
        t0 = time.time()
        np.savez_compressed(arq, **estatisticas_zscore(frases_z, tok, mod))
        registro.append({"modelo": curto, "etapa": "zscore", "frases": len(frases_z), "segundos": round(time.time() - t0)})

    # 3. Jeito 1 (só na ETAPA "teste"): todas as frases de 2012 e 2018
    if ETAPA == "teste":
        for ano, caminho in completos.items():
            arq = os.path.join(pasta, f"completo_{ano}.npz")
            if os.path.exists(arq): continue
            df = pd.read_parquet(caminho)
            if CONFERENCIA: df = df.head(2000)
            t0 = time.time()
            res, perdidos = vetores_por_palavra(df, tok, mod, f"{curto} completo {ano}")
            np.savez_compressed(arq, **res)
            registro.append({"modelo": curto, "etapa": f"completo {ano}", "frases": len(df),
                             "palavras": len(res["palavras"]), "perdidos": perdidos, "segundos": round(time.time() - t0)})
            del df

    del mod; torch.cuda.empty_cache()
    pd.DataFrame(registro).to_csv(os.path.join(SAIDA, "registro.csv"), index=False)

print("\n================ RESUMO ================")
print(pd.DataFrame(registro).to_string(index=False))

In [ ]:
# Conferência rápida dos arquivos gerados
for arq in sorted(glob.glob(os.path.join(SAIDA, "*", "*.npz"))):
    d = np.load(arq)
    if "medias" in d:
        m = d["medias"]
        print(f"{os.path.relpath(arq, SAIDA):<40} palavras={len(d['palavras']):>4}  dim={m.shape[1]}  "
              f"NaN={int(np.isnan(m).sum())}  frases por palavra: mediana {int(np.median(d['n']))}, mín {int(d['n'].min())}")
    else:
        print(f"{os.path.relpath(arq, SAIDA):<40} z-score com {int(d['n'])} frases, dim={len(d['media'])}")

# Um único arquivo .zip para baixar
shutil.make_archive(SAIDA, "zip", SAIDA)
print("\nPara baixar:", SAIDA + ".zip")